# 5장 — 보상으로 배우기 🎯

웹 챕터의 실험에서는 "이번에 생성된 답"을 여러분이 직접 골랐죠.
진짜 강화학습은 **정책이 스스로 샘플링**합니다. 주사위를 굴려 답을 뽑고, 점수를 받고, 확률을 고칩니다.

이번 노트북은 그 진짜 루프(REINFORCE)를 돌립니다:
**샘플 → 보상 → "잘한 행동은 더 자주"** 를 수백 번 반복하면, 정답을 알려준 적이 없는데도 정책이 좋은 답으로 수렴해요.

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 오른쪽 아래 버튼 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch5')

## 1. 무대 설치: 정책과 보상 모델

- 답변 후보 3개(웹 실험과 동일)와 보상 모델의 점수.
- 정책 = 소프트맥스(z): 점수판 z 를 확률분포로 바꿉니다(4부 9장에서 배운 그것). 처음엔 z 가 전부 0 → 균등 분포.

In [ ]:
import math
import random

random.seed(42)  # 재현 가능하게

ACTIONS = ['A 간결·정중', 'B 장황', 'C 무성의']
REWARDS = [3.0, 1.0, -2.0]  # 보상 모델(4장)의 점수

z = [0.0, 0.0, 0.0]  # 정책의 로짓(점수판) — 이게 우리의 '가중치'

def policy():
    """softmax(z) — 로짓을 확률분포로."""
    exps = [math.exp(v) for v in z]
    total = sum(exps)
    return [e / total for e in exps]

def expected_reward():
    return sum(p * r for p, r in zip(policy(), REWARDS))

print('초기 정책:', [f'{p:.0%}' for p in policy()])
print(f'초기 기대 보상 E[r] = {expected_reward():.2f}')

## 2. REINFORCE 루프: 샘플 → 보상 → 확률 조정

한 에피소드:
1. 정책에서 답변 하나를 **샘플링**(주사위)
2. 보상 모델에게 **보상**을 받고
3. **갱신**: 뽑힌 행동의 로짓은 `보상 × (1−π)` 만큼, 나머지는 `보상 × (−π)` 만큼 조정
   곧 "잘한 행동은 더 자주(+), 못한 행동은 덜 자주(−)"의 수학 버전이에요.
   (이 조정량이 바로 log π 의 기울기. 2부 9장의 경사 이동이고, 방향은 오르막)

In [ ]:
LR = 0.05

def episode():
    """샘플 → 보상 → 정책 갱신. 뽑힌 행동 번호를 돌려준다."""
    probs = policy()
    i = random.choices(range(len(ACTIONS)), weights=probs)[0]  # 1. 샘플
    r = REWARDS[i]                                             # 2. 보상
    for j in range(len(z)):                                    # 3. 갱신 (REINFORCE)
        grad_log_pi = (1 - probs[j]) if j == i else (-probs[j])
        z[j] += LR * r * grad_log_pi
    return i

curve = [expected_reward()]
picks = [0, 0, 0]
for _ in range(300):
    picks[episode()] += 1
    curve.append(expected_reward())

print('300 에피소드 동안 뽑힌 횟수:', dict(zip(ACTIONS, picks)))
print('학습 후 정책:', [f'{p:.0%}' for p in policy()])
print(f'학습 후 기대 보상 E[r] = {expected_reward():.2f}  (초기 {curve[0]:.2f})')

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 3))
plt.plot(range(len(curve)), curve, color='#6366f1', linewidth=1)
plt.axhline(max(REWARDS), color='#10b981', linestyle='--', linewidth=1, label='최선의 답만 낼 때 (+3)')
plt.xlabel('에피소드')
plt.ylabel('기대 보상 E[r]')
plt.title('REINFORCE: 정답 없이 점수만으로 오르는 언덕')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. 검증: 그리고 '나쁜 경험도 배움'

곡선이 +3(최선의 답만 낼 때) 쪽으로 기어 올라갔죠. 아무도 "정답은 A"라고 말해 준 적 없습니다.

마지막으로 웹 실험의 관찰 하나를 코드로 다시 확인합니다: **나쁜 답을 뽑은 에피소드도 E[r]을 올린다**.

In [ ]:
# 검증 1 — 학습으로 기대 보상이 크게 올랐다
assert curve[-1] > curve[0] + 1.5, '학습 후 E[r]이 충분히 올라야 한다!'
# 검증 2 — 정책이 최고 보상 답(A)에 수렴했다
assert policy()[0] == max(policy()), '최고 보상 행동의 확률이 가장 높아야 한다!'

# 검증 3 — 나쁜 경험의 가치: 균등 정책에서 C(보상 −2)를 강제로 겪게 하면?
z2 = [0.0, 0.0, 0.0]
probs = [1 / 3] * 3
before = sum(p * r for p, r in zip(probs, REWARDS))
for j in range(3):  # C(인덱스 2)를 뽑았다고 치고 REINFORCE 갱신
    grad = (1 - probs[j]) if j == 2 else (-probs[j])
    z2[j] += 0.5 * REWARDS[2] * grad
exps = [math.exp(v) for v in z2]
probs2 = [e / sum(exps) for e in exps]
after = sum(p * r for p, r in zip(probs2, REWARDS))
print(f'C(무성의, −2)를 겪은 뒤: C 확률 {probs[2]:.0%} → {probs2[2]:.0%}, E[r] {before:.2f} → {after:.2f}')
assert after > before, '나쁜 경험도 E[r]을 올려야 한다 (그쪽은 아니야를 배움)!'
print('\n✔ 확인: 언덕 오르기 성공 + 실패도 정보다')

## 4. 관찰 정리 + 🤖 AI 연결

- **정책 = softmax(점수판)**(4부 9장), **목표 = E[r]**(3부 5장 기댓값), **갱신 = 경사 이동, 오르막**(2부 9장).
- 갱신량 `보상 × (1−π 또는 −π)`: 잘한 행동은 더 자주, 못한 행동은 덜 자주의 수식 버전.
- 진짜 RLHF와 다른 점: 진짜는 답변 3개가 아니라 **토큰 시퀀스 전체**가 행동이고,
  갱신도 신경망 역전파로 합니다. 하지만 뼈대는 이 루프 그대로입니다.
- 그리고 이 루프에는 아직 안전장치가 없어요. 보상 모델의 빈틈을 파고들기 시작하면? 6장에서 봅니다.

**다음 챕터**: 6장, RLHF 조립 + KL 목줄(5부 5장의 KL 발산이 주인공으로 귀환).

> 💡 궁금한 코드는 그 셀을 실행한 다음 셀에서 `from tutor import explain` 후 `explain('ch5')`.